In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum,avg,count

In [3]:
spark=SparkSession.builder.appName("GroupByExample").getOrCreate()

In [4]:
data=[
    ("Akash","Sales",10000),
    ("Siya","Sales",12000),
    ("Tarun","IT",22000),
    ("Joshika","IT",18000),
    ("Ian","HR",13000)
]

columns=["Name","Department","Salary"]

In [5]:
df=spark.createDataFrame(data,columns)
df.show()

+-------+----------+------+
|   Name|Department|Salary|
+-------+----------+------+
|  Akash|     Sales| 10000|
|   Siya|     Sales| 12000|
|  Tarun|        IT| 22000|
|Joshika|        IT| 18000|
|    Ian|        HR| 13000|
+-------+----------+------+



In [6]:
#Group By Department and Sum Salary
df.groupBy("Department").agg(sum("Salary").alias("Total_Salary")).show()

+----------+------------+
|Department|Total_Salary|
+----------+------------+
|     Sales|       22000|
|        HR|       13000|
|        IT|       40000|
+----------+------------+



In [7]:
#GroupBY Department and Get multiple aggregations
df.groupBy("Department").agg(
    sum("Salary").alias("Total_Salary"),
    avg("Salary").alias("Anerage_Salary"),
    count("Name").alias("Employee_Count")
).show()

+----------+------------+--------------+--------------+
|Department|Total_Salary|Anerage_Salary|Employee_Count|
+----------+------------+--------------+--------------+
|     Sales|       22000|       11000.0|             2|
|        HR|       13000|       13000.0|             1|
|        IT|       40000|       20000.0|             2|
+----------+------------+--------------+--------------+



In [8]:
#Group by Multiple columns
df.groupBy("Department","Name").agg(sum("Salary")).show()

+----------+-------+-----------+
|Department|   Name|sum(Salary)|
+----------+-------+-----------+
|     Sales|   Siya|      12000|
|     Sales|  Akash|      10000|
|        IT|Joshika|      18000|
|        IT|  Tarun|      22000|
|        HR|    Ian|      13000|
+----------+-------+-----------+



In [9]:
df.groupBy("Name").agg(avg("Salary")).show()

+-------+-----------+
|   Name|avg(Salary)|
+-------+-----------+
|   Siya|    12000.0|
|  Akash|    10000.0|
|Joshika|    18000.0|
|  Tarun|    22000.0|
|    Ian|    13000.0|
+-------+-----------+



In [10]:
#Collecting data into lists oe sets
from pyspark.sql.functions import collect_list, collect_set

df.groupBy("Department").agg(
    collect_list("Name").alias("Employees"),
    collect_set("Salary").alias("Unique_Salary")
).show()

#You can also collet values into a list or set per group

+----------+----------------+--------------+
|Department|       Employees| Unique_Salary|
+----------+----------------+--------------+
|     Sales|   [Akash, Siya]|[10000, 12000]|
|        HR|           [Ian]|       [13000]|
|        IT|[Tarun, Joshika]|[18000, 22000]|
+----------+----------------+--------------+



In [11]:
#Dataset with Null values
spark=SparkSession.builder.appName("GroupWithNullHandling").getOrCreate()

In [12]:
data=[
    ("Akash","Sales",10000),
    ("Siya","Sales",12000),
    ("Tarun","IT",None),
    ("Ritik","IT",18000),
    ("Ian","HR",13000),
    ("Aman",None,23000),
    (None,"Sales",15000),
    ("Shreya","HR",20000),
    ("Karan",None,13000),
    ("Parth","IT",None)
]

columns=["Name","Department","Salary"]

In [13]:
df=spark.createDataFrame(data,columns)
df.show()

+------+----------+------+
|  Name|Department|Salary|
+------+----------+------+
| Akash|     Sales| 10000|
|  Siya|     Sales| 12000|
| Tarun|        IT|  NULL|
| Ritik|        IT| 18000|
|   Ian|        HR| 13000|
|  Aman|      NULL| 23000|
|  NULL|     Sales| 15000|
|Shreya|        HR| 20000|
| Karan|      NULL| 13000|
| Parth|        IT|  NULL|
+------+----------+------+



In [14]:
#Removing the null value
df_clean=df.na.drop(subset=["Department"])
df_clean.show()

+------+----------+------+
|  Name|Department|Salary|
+------+----------+------+
| Akash|     Sales| 10000|
|  Siya|     Sales| 12000|
| Tarun|        IT|  NULL|
| Ritik|        IT| 18000|
|   Ian|        HR| 13000|
|  NULL|     Sales| 15000|
|Shreya|        HR| 20000|
| Parth|        IT|  NULL|
+------+----------+------+



In [15]:
#Filling missing values with 0 or average

df_clean=df_clean.na.fill({"Salary":0})
df_clean.show()

+------+----------+------+
|  Name|Department|Salary|
+------+----------+------+
| Akash|     Sales| 10000|
|  Siya|     Sales| 12000|
| Tarun|        IT|     0|
| Ritik|        IT| 18000|
|   Ian|        HR| 13000|
|  NULL|     Sales| 15000|
|Shreya|        HR| 20000|
| Parth|        IT|     0|
+------+----------+------+



In [16]:
columns=["Name","Department","Salary"]
df=spark.createDataFrame(data,columns)

#Step1: Drop rows where department is nul
df_clean=df.na.drop(subset=["Department"])

#Step2: Fill missing salary with 0 or average
avg_salary=df.select(avg("Salary")).collect()[0][0]
df_clean=df_clean.na.fill({"Salary":avg_salary})


# Step 3: Group by Department
result = df_clean.groupBy("Department").agg(
    sum("Salary").alias("Total_Salary"),
    avg("Salary").alias("Avg_Salary"),
    count("Name").alias("Num_Employees")
)

result.show()

+----------+------------+------------------+-------------+
|Department|Total_Salary|        Avg_Salary|Num_Employees|
+----------+------------+------------------+-------------+
|     Sales|       37000|12333.333333333334|            2|
|        HR|       33000|           16500.0|            2|
|        IT|       49000|16333.333333333334|            3|
+----------+------------+------------------+-------------+



#NULL & NOTNULL

In [17]:
data=[
    ("Akash",10000),
    ("Siya",12000),
    ("Tarun",None),
    ("Joshika",18000),
    ("Ian",13000)
]

columns=["Name","Salary"]

In [18]:
df=spark.createDataFrame(data,columns)
df.show()

+-------+------+
|   Name|Salary|
+-------+------+
|  Akash| 10000|
|   Siya| 12000|
|  Tarun|  NULL|
|Joshika| 18000|
|    Ian| 13000|
+-------+------+



In [19]:
#Filter rows with all
df.filter(df.Salary.isNull()).show()

+-----+------+
| Name|Salary|
+-----+------+
|Tarun|  NULL|
+-----+------+



In [21]:
#Filter rows with not null
df.filter(df.Salary.isNotNull()).show()

+-------+------+
|   Name|Salary|
+-------+------+
|  Akash| 10000|
|   Siya| 12000|
|Joshika| 18000|
|    Ian| 13000|
+-------+------+

